In [0]:
%pip install -q pystac-client planetary-computer rasterio

In [0]:
dbutils.library.restartPython()

In [0]:
import pystac_client
from pyspark.sql.functions import current_timestamp

bbox_brasil = [
    -74.0,   # min_lon
    -34.0,   # min_lat
    -34.0,   # max_lon
     5.5     # max_lat
]

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)

search = catalog.search(
    collections=["landsat-c2-l2"],
    bbox=bbox_brasil,
    datetime="2025-01-01/2025-12-31",
    query={
        "eo:cloud_cover": {
            "lt": 20
        }
    }
)

dados = []

for item in search.items():

    dados.append({
        "scene_id": item.id,
        "datetime": str(item.datetime),
        "collection": item.collection_id,
        "cloud_cover": item.properties.get("eo:cloud_cover"),
        "platform": item.properties.get("platform"),
        "instruments": str(item.properties.get("instruments")),
        "bbox": str(item.bbox)
    })

df_bronze = (
    spark.createDataFrame(dados)
         .withColumn("ingestion_timestamp", current_timestamp())
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gs_carbon.bronze.landsat_metadata")
)

display(df_bronze)